# Model Selection Report — [dataset name]

- **Group**: [names and student IDs]
- **Variant**: binary classification
- **Dataset**: [name, with a link to its source]
- **Date**: [YYYY-MM-DD]

> *Fill in the bracketed fields above, then delete this quoted line.*

🗑️ **Delete this section before submitting** — it is instructions, not part of your report.

## Start Here

This notebook is a **complete worked example** of one workflow: choosing, tuning and testing a
classification model. It runs on a customer churn dataset. Your group repeats the same workflow on a
dataset you chose yourselves. **Copy this file, keep its structure, and replace the churn content
with yours.** The copy is what you hand in.

**What you will practise:**

1. Describe a dataset and check that it fits the task.
2. Choose one metric before seeing any result, and use it everywhere.
3. Split the data into a training set and a test set, and define the cross-validation folds once.
4. Build a preprocessing pipeline that learns only from the rows it is fitted on.
5. Compare a baseline and four model families with cross-validation, and choose one by a rule.
6. Tune the chosen model with the same folds and the same metric.
7. Measure the final model once on the test set, and compare that number with the estimate.

**The workflow, in one picture:**

```
data ─► split ─┬─► training set ─► 5-fold CV ─► tournament ─► chosen model ─► tuning ─┐
               │                   (baseline + 4 families)     (same CV)              │
               └─► test set ── kept aside, not used ─────────────────────► one final test
```

**Your report is eight ✍️ cells**, one per step:

| # | Deliverable | Section |
|---|---|---|
| 1 | Dataset card | 1 |
| 2 | The metric | 2 |
| 3 | The split | 3 |
| 4 | The preprocessing table | 4 |
| 5 | The tournament | 5 |
| 6 | The tuning | 6 |
| 7 | The final test | 7 |
| 8 | Conclusions | 8 |

**What each kind of cell is, and what you do with it:**

| Cell | What it is | What you do |
|---|---|---|
| ✍️ **Deliverable** | a question for your group | answer it about **your** data; keep it |
| 💡 **Example** | our answer for the churn data | read it, then delete it |
| 🗑️ **Guide** | what to do in a section, and 🎯 what we check | read it, then delete it |
| code cells | the workflow | keep them; change them for your data |
| other markdown | the sentence before each code cell, and the reading after it | keep them; rewrite them about your data and your numbers |

**Rules that hold in every section.** Each one is checked when we grade:

1. **The test set is used once**, in Section 7, and never before.
2. **One metric** is used for every comparison: tournament, tuning and test.
3. **One random seed**, set in Section 0, is passed to every step that uses randomness.
4. **Every number in your text comes from an output in your notebook.** Do not copy our numbers.
5. **Every code cell has a sentence before it** saying what the cell does.
6. **Run the whole notebook from top to bottom** (*Restart Kernel and Run All Cells*) before you
   submit, and submit it **with its outputs**.

**Your dataset must meet these minimum requirements.** Section 1 checks them with code:

- a **binary target** (two classes), with the smaller class at **20% of the rows or more**;
- at least **1,000 rows**;
- at least **8 predictor columns**, with both numeric and categorical columns among them;
- at least one real preprocessing need beyond scaling, such as missing values or categorical columns
  to encode;
- **not** one of the course datasets (churn, Ames housing, iris, insurance, auto MPG).

Two groups may use the same dataset, at their own risk.

**A limitation you know in advance.** The tournament in Section 5 compares models at **starting
settings** fixed by simple rules, not at tuned ones, and only the winner is tuned. A model can still
lose a comparison it might win after tuning. This is accepted on purpose, to keep the workflow short.
Say so in your conclusions; you do not need to fix it.

**How your work is graded.** Each ✍️ cell is judged on your own dataset. We do **not** compare scores
between groups. A modest score on a hard problem, reported honestly, earns more than a high score
that comes from a mistake. Each section's 🗑️ guide ends with a 🎯 list: that list is what we look
for in that section.

## 0. Setup

🗑️ **Guide — Section 0.** Delete before submitting.

Put every import in the one cell below, never in the middle of the notebook. Set the random seed
once, here, and pass it to every step that uses randomness.

🔧 **For your data:** add any import you need here. Keep the seed.

🎯 **What we check:** one import cell at the top; one seed, defined once and used in every random step.

The cell below imports every library the notebook uses and fixes the random seed. If it runs
without an error, the environment is ready:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay, roc_auc_score
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

## 1. The Data

🗑️ **Guide — Section 1.** Delete before submitting.

This section loads the data, makes the structural fixes, looks at the target and checks the minimum
requirements.

- **Before the split**, make only fixes that compute nothing from the data: drop identifiers, recode
  the target to 1/0, turn hidden missing markers (a blank, a `"?"`) into real missing values, and fix
  column types.
- **Leave for the pipeline in Section 4** every step that computes a value from the data: filling
  missing values, scaling and encoding. Computed here, on all the rows, they would use information
  from the test set. That is **leakage**, and it makes every score in the notebook too optimistic.
- **Drop leaking columns**: columns only known after the outcome, such as a cancellation date when
  you predict cancellation. A model with a leaking column scores very well here and fails in real
  use. Name them in Deliverable 1.

🔧 **For your data:** the file path, the columns you drop, the structural fixes and the name of the
target column.

🎯 **What we check:**
- Deliverable 1 answers every point, about your data.
- The requirements check prints `PASS` on every line.
- Identifiers and leaking columns are dropped, and named with a reason.
- No step before the split computes a value from the data.

### 1.1 Loading the raw file

The cell below reads the raw file and prints its size:

In [ ]:
DATA_PATH = "../data/telco_churn.csv"

raw = pd.read_csv(DATA_PATH)
print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")

The data has 7,043 rows and 21 columns. **One row is one customer** of a telecommunications company.
The next cell shows the first rows, so you can see what a customer looks like in the data:

In [ ]:
raw.head()

Most columns are text: yes/no answers and service types. The next cell lists each column's type,
its number of missing values and its number of distinct values:

In [ ]:
pd.DataFrame(
    {
        "type": raw.dtypes.astype(str),
        "missing": raw.isna().sum(),
        "distinct values": raw.nunique(),
    }
)

Two things to read here. `customerID` has 7,043 distinct values, one per row: it is an
**identifier**, not information about the customer. `TotalCharges` is an amount of money, but it was
read as **text**, and no column reports a missing value. A number column read as text usually hides
values that are not numbers.

### 1.2 Structural fixes

The next cell finds the values of `TotalCharges` that cannot be read as numbers, and shows those rows
next to `tenure`, the number of months the customer has been with the company:

In [ ]:
not_a_number = pd.to_numeric(raw["TotalCharges"], errors="coerce").isna()

print(f"rows where TotalCharges is not a number: {not_a_number.sum()}")
raw.loc[not_a_number, ["tenure", "MonthlyCharges", "TotalCharges"]]

Eleven rows hold a blank space instead of a number, and every one of them has `tenure` 0: they are new
customers who have not received a bill yet. The blank is a **hidden missing value**. We turn it into a
real missing value here, because that learns nothing from the data. **How to fill it is a
preprocessing decision**, and it belongs to the pipeline in Section 4.

The cell below makes the three structural fixes: it drops the identifier, turns the blanks into
missing values and converts the column to numbers, and turns the target into 1 (left) and 0 (stayed):

In [ ]:
TARGET = "Churn"

data = raw.drop(columns="customerID")
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"].str.strip().replace("", np.nan))
data[TARGET] = (data[TARGET] == "Yes").astype(int)

print(f"{data.shape[0]:,} rows x {data.shape[1]} columns")
print(f"missing values in TotalCharges: {data['TotalCharges'].isna().sum()}")

There are now 20 columns: 19 predictors and the target. The 11 missing values are visible, so the
pipeline can deal with them.

> **In `pandas`.** `pd.to_numeric(..., errors="coerce")` turns anything it cannot read into a
> missing value **without saying so**. We used it above only to *find* the bad values. For the fix
> itself we replaced the one value we understood (the blank) and converted without `errors=`, so that
> any other unexpected text would stop the notebook with an error instead of disappearing.

### 1.3 The target

The next cell counts the customers in each class of the target:

In [ ]:
counts = data[TARGET].value_counts()
pd.DataFrame({"customers": counts, "share": (counts / len(data)).round(4)})

26.5% of customers left: that is class 1, the positive class. The other 73.5% stayed.

### 1.4 Minimum requirements check

The cell below checks each minimum requirement from *Start Here* and prints `PASS` or `FAIL`. Keep it
in your report: it is the evidence for Deliverable 1.

In [ ]:
predictors = data.drop(columns=TARGET)
n_numeric = predictors.select_dtypes(include="number").shape[1]
n_categorical = predictors.shape[1] - n_numeric
minority_share = data[TARGET].value_counts(normalize=True).min()
n_missing = int(predictors.isna().sum().sum())

requirements = {
    "binary target": data[TARGET].nunique() == 2,
    f"smaller class >= 20% (it is {minority_share:.1%})": minority_share >= 0.20,
    f"at least 1,000 rows (it has {len(data):,})": len(data) >= 1000,
    f"at least 8 predictors (it has {predictors.shape[1]})": predictors.shape[1] >= 8,
    f"numeric and categorical columns ({n_numeric} and {n_categorical})": n_numeric > 0 and n_categorical > 0,
    f"a preprocessing need ({n_missing} missing, {n_categorical} to encode)": n_missing > 0 or n_categorical > 0,
}

for requirement, passed in requirements.items():
    print(f"{'PASS' if passed else 'FAIL'}  {requirement}")

Every line passes. `SeniorCitizen` counts as numeric here because it is stored as 0/1, although it is
really a yes/no column. Section 4 gives it its own treatment.

✍️ **Deliverable 1 of 8 — Dataset card**

- **Name and source** (with a link): *...*
- **What one row represents**: *...*
- **Size after the structural fixes** (rows × columns): *...*
- **The target**: its column name, what 1 means, what 0 means, and the share of the positive class: *...*
- **The decision a model could support**, and who would use it: *...*
- **Columns dropped before modelling**, each with its reason (identifiers, leaking columns): *...*
- **Structural fixes made before the split**, and why each one learns nothing from the data: *...*
- **Requirements check**: confirm that every line above prints `PASS`: *...*

💡 **Example** — Deliverable 1 for the churn data. Delete it before submitting.

- **Name and source**: *Telco Customer Churn*, a sample dataset published by IBM and distributed on
  Kaggle (`kaggle.com/datasets/blastchar/telco-customer-churn`).
- **What one row represents**: one customer of a telecommunications company: who they are, which
  services they buy and how their account is billed.
- **Size after the structural fixes**: 7,043 rows × 20 columns (19 predictors and the target).
- **The target**: `Churn`. 1 means the customer left in the last month, 0 means they stayed. 26.5% of
  customers are positive.
- **The decision**: which customers the retention team should contact first with an offer. The model
  ranks customers by their risk of leaving.
- **Columns dropped**: `customerID`, an identifier with one value per row; a model could only
  memorize it. We checked for leaking columns: none records something that happens after a customer
  leaves. `TotalCharges` grows with `tenure`, but both are known before the customer leaves, so it is
  history, not leakage.
- **Structural fixes**: blanks in `TotalCharges` turned into missing values and the column converted
  to numbers; the target turned from `Yes`/`No` into 1/0. Neither step computes anything from other
  rows.
- **Requirements check**: every line prints `PASS`.

## 2. The Metric

🗑️ **Guide — Section 2.** Delete before submitting.

Choose the metric before any model is trained, and use the same one in every section. For this
variant the metric is **ROC AUC**. In Deliverable 2, explain why leaving the threshold for later fits
your decision.

🔧 **For your data:** nothing in the code. The justification in Deliverable 2 is yours.

🎯 **What we check:**
- One metric, stored once and used in every scoring step. Search your notebook for `scoring=`.
- A justification that names what **your** threshold would depend on, and why that is not known yet
  or not the modelling team's decision.
- A justification that talks about your target and your decision, not about metrics in general.

### Why ROC AUC

A classifier gives every row a **score**: a number that rises with the chance of the positive class,
such as a predicted probability. Turning scores into *leaves / stays* predictions needs a
**threshold**, and metrics such as accuracy, precision, recall and F1 all change with where it is set.
Choosing it needs costs we do not have yet (the price of an offer, the value of a customer kept), and
it is a separate decision from choosing a model. So we use a metric that judges the scores
themselves, across every threshold at once. **ROC AUC** does that. It ranges from 0 to 1.

> **In `scikit-learn`.** `scoring="roc_auc"`; the scores come from `predict_proba` or
> `decision_function`.

The cell below stores the metric's name in one constant. Every later scoring step uses it, so the
metric is set in one place only:

In [ ]:
SCORING = "roc_auc"

✍️ **Deliverable 2 of 8 — The metric**

- **The metric**: *...*
- **What a score of 0.5 and a score of 1.0 would mean for your problem**, in words about your rows: *...*
- **What a threshold would mean in your problem**: what happens to a row the model flags, and what
  happens to one it does not: *...*
- **What the threshold would depend on**, and why it is left for later: *...*

💡 **Example** — Deliverable 2 for the churn data. Delete it before submitting.

- **The metric**: ROC AUC, used in the tournament, the tuning and the final test.
- **0.5 and 1.0**: 0.5 means the model's risk ranking puts leaving customers above staying ones no
  more often than a coin would. 1.0 means every customer who left is ranked above every customer who
  stayed.
- **What a threshold would mean**: a flagged customer receives a retention offer; a customer who is
  not flagged is left alone.
- **What it depends on**: the cost of an offer, the value of a customer kept, and the retention
  team's budget. None of these is known yet, and the budget is the business's decision, not ours.
  The team will contact customers from the top of the ranking down until the budget runs out, so the
  model's job is the ranking, and AUC measures exactly that.

## 3. Splitting the Data

🗑️ **Guide — Section 3.** Delete before submitting.

This section makes both splits, in one place:

- **The hold-out test set**: 20% of the rows, stratified on the target and seeded. It is not used
  again until Section 7: every earlier look at it makes its final score less honest.
- **The cross-validation folds**: one `StratifiedKFold` object, shuffled and seeded, passed as `cv=`
  to every cross-validation step in Sections 5 and 6, so every model is scored on the same rows.

🔧 **For your data:** usually nothing. Keep 20% for the test set and 5 folds, stratified.

🎯 **What we check:**
- The split happens before any step that learns from data.
- Both splits are stratified and seeded.
- One fold object, created here and passed to **every** cross-validation step.
- The class share printed for the training and test sets.
- The test set is not used again until Section 7.

### 3.1 The hold-out test set

The cell below separates the predictors from the target and sets 20% of the rows aside as the test
set. Stratifying on the target keeps the share of churners equal in both parts:

In [ ]:
X = data.drop(columns=TARGET)
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

The next cell checks the size of each part and its share of positive rows:

In [ ]:
pd.DataFrame(
    {
        "rows": [len(y_train), len(y_test)],
        "positive share": [y_train.mean(), y_test.mean()],
    },
    index=["training set", "test set"],
).round(4)

The training set has 5,634 rows and the test set 1,409, and both keep the 26.5% share of churners.
**From here until Section 7, `X_test` and `y_test` are not used.**

### 3.2 The cross-validation folds

> **In `scikit-learn`.** `StratifiedKFold` defines stratified folds. `shuffle=True` mixes the rows
> before they are divided, so that any order in the file (for example, by date) does not decide the
> folds; shuffling uses randomness, so it takes the seed. The object is then passed as `cv=` to every
> cross-validation function.

The cell below creates the fold object. It is created here once and passed to every cross-validation
step in Sections 5 and 6:

In [ ]:
N_FOLDS = 5
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

The next cell lists the five folds, with the number of rows each model is fitted on, the size of the
validation fold and its share of positive rows:

In [ ]:
fold_rows = []
for fold, (fit_index, validation_index) in enumerate(cv.split(X_train, y_train), start=1):
    fold_rows.append(
        {
            "fold": fold,
            "rows fitted on": len(fit_index),
            "validation rows": len(validation_index),
            "validation positive share": y_train.iloc[validation_index].mean(),
        }
    )

pd.DataFrame(fold_rows).set_index("fold").round(4)

Every model is fitted on about 4,507 rows and scored on about 1,127, and every validation fold keeps
the 26.5% share. The five folds are the same for every model from here on.

✍️ **Deliverable 3 of 8 — The split**

- **Sizes**: rows in the training set and in the test set: *...*
- **Why this share for the test set**: *...*
- **Number of folds, and the rows each model is fitted on and scored on in one fold**: *...*
- **Why the splits are stratified**, with your positive share: *...*
- **The rule you follow for the test set from now on**: *...*

💡 **Example** — Deliverable 3 for the churn data. Delete it before submitting.

- **Sizes**: 5,634 rows for training, 1,409 for the test set.
- **Why 20%**: 1,409 test rows, 26.5% of them customers who left, are enough for one stable final
  score, and 80% of the rows stay for training.
- **Folds**: 5 folds. Each model is fitted on about 4,507 rows and scored on about 1,127.
- **Why stratified**: churners are 26.5% of the rows. Stratifying keeps that share in the test set
  and in every fold, so no score moves because a part got more or fewer churners by chance.
- **The test set rule**: `X_test` and `y_test` are used once, in Section 7, to score the final model.

## 4. The Preprocessing Pipeline

🗑️ **Guide — Section 4.** Delete before submitting.

- Put every step that computes a value from the data (imputers, scalers, encoders) in the
  preprocessor, and join it to each model with `make_model`. Cross-validation then refits it on the
  fitting folds of every round.
- Never pass a preprocessed matrix to cross-validation: always the raw `X_train` and the full
  pipeline. Preprocessing the whole training set first lets every validation fold shape the medians
  and means the model is trained with. That is leakage; its effect is often small, so it goes
  unnoticed.
- Use one preprocessor for every candidate.
- Scale the numeric columns: KNN and logistic regression need it.

🔧 **For your data:** the column groups and the treatment of each group. Common extra treatments:
- a log transform for a strongly skewed numeric column;
- an ordered encoding for categories that have a natural order;
- grouping rare categories in a column with very many categories.

Every treatment you add must be justified in Deliverable 4.

🎯 **What we check:**
- Every predictor column in exactly one group (the check cell passes).
- Preprocessing only inside the pipeline: no `fit_transform` whose result goes into cross-validation.
- Nothing fitted on the test set.
- Deliverable 4 covers every group and justifies every treatment with your data.

### 4.1 Column groups

The cell below assigns every predictor column to one group, by name:

In [ ]:
NUMERIC_COLS = ["tenure", "MonthlyCharges", "TotalCharges"]
BINARY_COLS = ["SeniorCitizen"]
CATEGORICAL_COLS = [
    column for column in X_train.columns if column not in NUMERIC_COLS + BINARY_COLS
]

print(f"numeric ({len(NUMERIC_COLS)}): {NUMERIC_COLS}")
print(f"binary ({len(BINARY_COLS)}): {BINARY_COLS}")
print(f"categorical ({len(CATEGORICAL_COLS)}): {CATEGORICAL_COLS}")

A column left out of every group would be **dropped without any warning**. The next cell checks that
each predictor is in exactly one group, and stops the notebook if one is not:

In [ ]:
assigned = NUMERIC_COLS + BINARY_COLS + CATEGORICAL_COLS

assert len(assigned) == len(set(assigned)), "a column is in more than one group"
assert set(assigned) == set(X_train.columns), "a column is in no group, or a name is misspelled"
print(f"all {len(assigned)} predictor columns are in exactly one group")

### 4.2 The treatment of each group

The conventional treatments are as follows:

- **Numeric columns.** Missing values are filled with the column's **median** in the fitting rows.
  Then each column is **standardized**.
  Standardized columns are 0 at the mean, and measured in standard deviations. These columns have
  no fixed range, but most values fall between −3 and 3.

- **Binary columns** are already 0/1 and are kept as they are.
- **Categorical columns.** Missing values are filled with the most frequent category. Then **one-hot
  encoding** turns each column into one 0/1 column per category: `Contract` becomes three columns,
  one for each contract type.

> **In `scikit-learn`.** `ColumnTransformer` sends each group to its own steps, and by default it
> **drops** any column not listed in a group (`remainder="drop"`); that is why 4.1 checks the groups.
> `SimpleImputer`, `StandardScaler` and `OneHotEncoder` do the steps above.
> `handle_unknown="ignore"` makes the encoder write all zeros for a category it did not see during
> fitting, instead of stopping with an error; a rare category can be absent from the fitting folds of
> one round. `set_output(transform="pandas")` keeps the column names in the output.

The cell below builds the preprocessor from the three groups. Nothing is fitted yet:

In [ ]:
numeric_steps = Pipeline(
    [
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
)
categorical_steps = Pipeline(
    [
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    [
        ("numeric", numeric_steps, NUMERIC_COLS),
        ("binary", "passthrough", BINARY_COLS),
        ("categorical", categorical_steps, CATEGORICAL_COLS),
    ],
    verbose_feature_names_out=False,
).set_output(transform="pandas")

The next cell defines `make_model`, which joins the preprocessor and a model into one pipeline.
Every model in Sections 5, 6 and 7 is built with it, so every model gets exactly the same
preprocessing:

In [ ]:
def make_model(estimator: BaseEstimator) -> Pipeline:
    """Join the shared preprocessor and a model into one pipeline.

    Args:
        estimator: Any scikit-learn classifier, not yet fitted.

    Returns:
        A pipeline that prepares the columns and then applies the model.
    """
    return Pipeline([("prep", preprocessor), ("model", estimator)])

The next cell fits the preprocessor on the training set **only to look at its output**: how many
columns the models will receive, and what they look like:

In [ ]:
X_train_preproc = preprocessor.fit_transform(X_train)

print(f"{X_train.shape[1]} columns in, {X_train_preproc.shape[1]} columns out")
X_train_preproc.head(3)

The 19 predictors become 45 columns, most of them one-hot columns from the 15 categorical predictors.
The numeric columns now have values close to 0.

>**`X_train_preproc` exists only to be looked at.** It was prepared with statistics from all of the
>training set, so it must never be passed to cross-validation. From Section 5 on, cross-validation
>always receives the raw `X_train` and the full pipeline.

✍️ **Deliverable 4 of 8 — The preprocessing table**

One row per column group:

| Group | Columns | Treatment, in order | Why, for your data |
|---|---|---|---|
| | | | |

Then answer:
- **Where your missing values are** and how the pipeline fills them: *...*
- **The number of columns** before and after preprocessing: *...*
- **Why the preprocessing is inside the pipeline**, in one or two sentences: *...*

💡 **Example** — Deliverable 4 for the churn data. Delete it before submitting.

| Group | Columns | Treatment, in order | Why, for your data |
|---|---|---|---|
| Numeric | `tenure`, `MonthlyCharges`, `TotalCharges` | median fill, then standardize | `TotalCharges` has 11 missing values, and the median is not pulled by the long right tail of charges; scaling puts months and money on one scale for KNN and logistic regression |
| Binary | `SeniorCitizen` | none | already 0/1 |
| Categorical | the other 15 columns | most-frequent fill, then one-hot | no missing values today, but the fill costs nothing and protects against a blank in new data; one-hot because the categories (contract type, payment method) have no order |

- **Missing values**: only the 11 new customers in `TotalCharges`. The pipeline fills them with the
  median of the fitting rows. Filling with 0 would also be defensible, since they have not been
  billed yet. With 11 rows out of 7,043, the choice cannot move the results.
- **Columns**: 19 predictors in, 45 columns out.
- **Why inside the pipeline**: cross-validation refits the pipeline in every round, so the medians,
  means and category lists come from the fitting folds only, and the validation fold stays unseen.

## 5. The Model Tournament

🗑️ **Guide — Section 5.** Delete before submitting.

Score every candidate with the same folds, the same metric and the same preprocessing.

- Include the baseline and one model from each of the four families.
- Start each family from the rule in the table below. The cell in 5.1 computes the values from your
  data size. Fix them before the tournament runs, and do not change them after seeing a score: that is tuning by
  hand, which nobody records and which makes the comparison with Section 6 unfair.
- The model with the highest mean CV AUC goes to Section 6.

**The four required families**, with their starting rules and the settings that matter most when
that family is tuned. $n$ is the number of rows each model is fitted on in one fold:

| Family | Our model | Starting setting | Main hyperparameters to tune |
|---|---|---|---|
| Regularized linear | logistic regression | library default ($C = 1$) | regularization strength; mix of L1 and L2 penalties |
| Nearest neighbours | KNN | $k = \sqrt{n}$, rounded to an odd number | number of neighbours; whether closer neighbours count more |
| Single tree | decision tree | maximum depth 5; at least 1% of $n$ rows per leaf | maximum depth; minimum rows per leaf |
| Ensemble | gradient boosting | library default (100 trees, depth 3, learning rate 0.1) | number of trees; learning rate; tree depth |

If your ensemble is a **random forest**, start it with at least 5 rows per leaf.

🔧 **For your data:** nothing in the rules; the starting-settings cell computes the values from
your row count. You may add candidates, for example a second ensemble, each with a stated starting
rule. You may not remove the baseline or any of the four families.

🎯 **What we check:**
- The baseline and the four families are all present.
- Every starting setting comes from a stated rule, computed in code, with no sign of changes made
  after seeing scores.
- The same fold object and the same metric in every call.
- A results table with the mean and the spread of the validation scores.
- The selection rule stated before the results.

### 5.1 The candidates and their starting settings

The starting rules for KNN and the decision tree need $n$, the number of rows each model is fitted on
in one fold. The cell below computes $n$ from the training set and the fold object, applies the two
rules, and prints the values the candidates will use:

In [ ]:
fit_rows = len(X_train) * (N_FOLDS - 1) // N_FOLDS

k_neighbors = round(np.sqrt(fit_rows))

# If k_neighbors is even -> k_neighbors + 1
if k_neighbors % 2 == 0:
    k_neighbors += 1
    
min_leaf = round(0.01 * fit_rows)

print(f"rows fitted on in one fold (n): {fit_rows:,}")
print(f"KNN: k = sqrt(n), rounded to an odd number -> {k_neighbors}")
print(f"decision tree: maximum depth 5, at least 1% of n per leaf -> {min_leaf} rows")

Each model is fitted on 4,507 rows, so KNN starts with $k = 67$ neighbours and the tree with at least
45 rows in every leaf. On a dataset of a different size, the same rules
give different values.

> **In `scikit-learn`.** `LogisticRegression` is regularized by default: an L2 penalty with strength
> set by `C=1.0`, where a **smaller** `C` means a **stronger** penalty. `max_iter=1000` gives the
> solver enough iterations to finish. `DummyClassifier(strategy="most_frequent")` is the baseline: it
> gives every row the same score, so its AUC is 0.5 by construction. For KNN, the number of neighbours
> is `n_neighbors`; for the tree, `max_depth` and `min_samples_leaf`.

The cell below lists the baseline and the four candidates, each at its starting setting:

In [ ]:
candidates = {
    "Baseline (most frequent class)": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=k_neighbors),
    "Decision tree": DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=min_leaf, random_state=RANDOM_STATE
    ),
    "Gradient boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

### 5.2 The selection rule

The rule is fixed before any score is seen: **the model with the highest mean CV AUC goes to
tuning.**

### 5.3 Running the tournament

> **In `scikit-learn`.** `cross_val_score` fits a fresh copy of the pipeline on each round's fitting
> folds and returns the validation scores, one per fold. `n_jobs=-1` runs the rounds in parallel on
> every processor core.

The cell below cross-validates every candidate with the fold object `cv` and the metric `SCORING`, and
keeps the five fold scores of each:

In [ ]:
cv_scores = {}
for name, estimator in candidates.items():
    cv_scores[name] = cross_val_score(
        make_model(estimator), X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1
    )

The next cell turns the fold scores into one table, with the mean and the standard deviation of each
model, sorted from best to worst mean:

In [ ]:
tournament = pd.DataFrame.from_dict(
    cv_scores,
    orient="index",
    columns=[f"auc_fold_{i}" for i in range(N_FOLDS)]
)
fold_columns = tournament.columns
tournament["cv_auc_mean"] = tournament[fold_columns].mean(axis=1)
tournament["cv_auc_std"] = tournament[fold_columns].std(axis=1)
tournament = tournament.sort_values("cv_auc_mean", ascending=False)
tournament

Every model is well above the baseline's 0.5. Gradient boosting has the highest mean, 0.8477,
followed by logistic regression at 0.8461, KNN at 0.8369 and the decision tree at 0.8303.

The next cell applies the selection rule and prints the model that goes to tuning:

In [ ]:
WINNER = tournament.index[0]
print(f"model chosen for tuning: {WINNER}")

Gradient boosting goes to Section 6.

✍️ **Deliverable 5 of 8 — The tournament**

- **Your starting settings**: for each family, the rule, the value it gave on your data, and a
  confirmation that no value was changed after seeing a score: *...*
- **Your selection rule**, as you applied it: *...*
- **The leader, and its margin over the next model**: *...*
- **The model you take to tuning**, and why: *...*
- **The baseline**: how far every model is above it: *...*

💡 **Example** — Deliverable 5 for the churn data. Delete it before submitting.

- **Starting settings**: logistic regression and gradient boosting at their library defaults; KNN
  with $k = \sqrt{n}$ rounded to odd, 67; the tree with depth 5 and at least 1% of $n$ per leaf, 45
  rows. All fixed before the tournament ran, and none changed afterwards.
- **Rule**: the highest mean CV AUC goes to tuning.
- **Leader**: gradient boosting, 0.8477, ahead of logistic regression (0.8461) by 0.0016.
- **Chosen**: gradient boosting, by the rule.
- **Baseline**: 0.5; the weakest real model, the decision tree, is 0.33 above it.

## 6. Tuning the Selected Model

🗑️ **Guide — Section 6.** Delete before submitting.

This section tunes the model chosen in Section 5 with an **Optuna** study, as in session 6. Every
trial is scored with the same `make_model`, the same fold object `cv` and the same `SCORING` as the
tournament.

The steps:

1. Write the objective: it builds the model from the values Optuna suggests and returns the mean CV
   score.
2. Run a study with a seeded sampler.
3. Compare each best value with its range. A value at an edge means the best may lie outside the
   range; widening it and running a new study is the fix.
4. Cross-validate the best settings and compare them with the untuned model.

**Starting ranges for each family**, to write your own objective:

| Family | Hyperparameter | Range | Optuna call |
|---|---|---|---|
| Logistic regression | `C` | 0.001 – 100 | `suggest_float(..., log=True)` |
| | `l1_ratio` | 0 – 1 | `suggest_float`; set `solver="saga"`, `max_iter=5000` |
| KNN | `n_neighbors` | 3 – 150 | `suggest_int` |
| | `weights` | `"uniform"`, `"distance"` | `suggest_categorical` |
| Decision tree | `max_depth` | 2 – 20 | `suggest_int` |
| | `min_samples_leaf` | 1 – 100 | `suggest_int` |
| Gradient boosting | `learning_rate` | 0.01 – 0.5 | `suggest_float(..., log=True)` |
| | `n_estimators` | 30 – 400 | `suggest_int` |
| | `max_depth` | 2 – 5 | `suggest_int` |
| | `subsample` | 0.6 – 1.0 | `suggest_float` |
| | `min_samples_leaf` | 1 – 60 | `suggest_int` |
| Random forest | `n_estimators` | 100 – 500 | `suggest_int` |
| | `max_features` | 0.1 – 1.0 | `suggest_float` |
| | `min_samples_leaf` | 1 – 50 | `suggest_int` |

🔧 **For your data:** the estimator and the `suggest_` lines in the objective, from the table. Keep
the seed, `cv`, `SCORING` and at least 30 trials.

🎯 **What we check:**
- The same fold object and the same metric as the tournament.
- A seeded sampler.
- Every range stated in Deliverable 6, with a reason.
- Best values at an edge of their range named in Deliverable 6.
- Tuned against untuned.
- The test set still unused.

### 6.1 The objective

> **In `optuna`.** `trial.suggest_float(name, low, high, log=True)` draws a decimal number on a log
> scale; `trial.suggest_int(name, low, high)` draws a whole number. Both include `low` and `high`.

The next cell defines the objective. Optuna calls it once per trial: it builds gradient boosting from
the suggested values and returns the mean CV AUC on the training set:

In [ ]:
def objective(trial: optuna.Trial) -> float:
    """Cross-validated AUC of gradient boosting for one set of hyperparameters.

    Args:
        trial: The Optuna trial that proposes the hyperparameter values.

    Returns:
        The mean ROC AUC across the folds of cv on the training set.
    """
    estimator = GradientBoostingClassifier(
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.5, log=True),
        n_estimators=trial.suggest_int("n_estimators", 30, 400),
        max_depth=trial.suggest_int("max_depth", 2, 5),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 60),
        random_state=RANDOM_STATE,
    )
    scores = cross_val_score(
        make_model(estimator), X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1
    )
    return scores.mean()

### 6.2 The study

The cell below creates the study: the direction to optimize, and the sampler with its seed fixed so
that the run repeats exactly:

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)

The next cell runs 30 trials, each one five fits. Optuna prints one line per trial as it finishes:

In [ ]:
N_TRIALS = 30

study.optimize(objective, n_trials=N_TRIALS)

The next cell prints the best score, the trial that reached it, and its settings:

In [ ]:
print(f"best CV AUC: {study.best_value:.4f} (trial {study.best_trial.number})")
pd.DataFrame([study.best_params])

The study reaches 0.8506, above the untuned 0.8477. The best trial is the last one,
number 29.

> Notice how `max_depth` sits at its lower edge, 2, so the best depth may lie below the range. One solution for this is running a new study that widens the depth range to start at 1.

### 6.3 Tuned against untuned

The cell below builds gradient boosting with the best settings and cross-validates it on the same
folds, next to the untuned row of the tournament:

In [ ]:
tuned_model = make_model(
    GradientBoostingClassifier(**study.best_params, random_state=RANDOM_STATE)
)
tuned_scores = cross_val_score(
    tuned_model, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1
)

pd.DataFrame(
    {
        "cv_auc_mean": [tournament.loc[WINNER, "cv_auc_mean"], tuned_scores.mean()],
        "cv_auc_std": [tournament.loc[WINNER, "cv_auc_std"], tuned_scores.std(ddof=1)],
    },
    index=["untuned (Section 5)", "tuned (Section 6)"],
).round(4)

Tuning raises the CV AUC from 0.8477 to 0.8506, a gain of 0.0029.

✍️ **Deliverable 6 of 8 — The tuning**

- **The objective**: each hyperparameter, its range and scale, and why: *...*
- **Trials and seed**: *...*
- **Edge check**: the values that sat at an edge, and what you did or would do: *...*
- **Tuned against untuned**: both scores and the gain: *...*

💡 **Example** — Deliverable 6 for the churn data. Delete it before submitting.

- **Objective**: gradient boosting, with the five ranges from the guide's table: `learning_rate`
  0.01–0.5 on a log scale, because useful values span more than one power of ten; `n_estimators`
  30–400; `max_depth` 2–5; `subsample` 0.6–1.0; `min_samples_leaf` 1–60.
- **Trials and seed**: 30 trials, TPE sampler seeded with `RANDOM_STATE` (42).
- **Edge check**: `max_depth` sits at its lower edge, 2. The next step would be a new study with the
  depth range starting at 1.
- **Tuned against untuned**: 0.8477 untuned, 0.8506 tuned, a gain of 0.0029.

## 7. The Final Test

🗑️ **Guide — Section 7.** Delete before submitting.

Use the test set once, here: refit the tuned model on the whole training set and score it.

Compare the test score with the cross-validated estimate:

- **within about one fold spread**: the estimate holds;
- **much lower**: check for leakage;
- **much higher**: the test rows may be easy by chance.

Do not change the model after seeing the test score: the test set would then have helped choose the
model, and its score would no longer be an honest estimate. If you do, say so in your conclusions.

🎯 **What we check:**
- One evaluation of the chosen model on the test set, after all choices are made. Search your
  notebook for `X_test`: it appears in Section 3 and here, nowhere else.
- The comparison with the cross-validated estimate.
- Deliverable 7 reads the gap correctly.

The cell below refits the tuned model on the whole training set, scores it once on the test set, and
prints the result next to the cross-validated estimate:

In [ ]:
final_model = tuned_model.fit(X_train, y_train)
test_scores = final_model.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, test_scores)

print("ROC AUC of the final model")
print(f"CV mean estimate: {tuned_scores.mean():.4f}")
print(f"CV spread: {tuned_scores.std(ddof=1):.4f}")
print(f"test set: {test_auc:.4f}")

The final model scores **0.8455** on the test set, against a cross-validated estimate of 0.8506
with a fold spread of 0.0135.

### The ROC curve

Each point of the **ROC curve** is one threshold. It plots the share of leaving customers that
threshold catches (the **true positive rate**) against the share of staying customers it flags by
mistake (the **false positive rate**). The AUC is the area under the curve; the diagonal is sorting
by chance, with an area of 0.5.

The next cell draws the final model's ROC curve on the test set, with the diagonal for reference:

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
RocCurveDisplay.from_predictions(y_test, test_scores, name="Final model", ax=ax)
ax.plot([0, 1], [0, 1], color="crimson", linestyle="--", linewidth=1, label="random sorting (AUC 0.5)")
ax.set_xlabel("false positive rate")
ax.set_ylabel("true positive rate")
ax.set_title("Final model on the test set")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

The curve rises steeply at first: the customers the model ranks highest are mostly real churners.
It then flattens, as the remaining churners are harder to separate from customers who stay. Where to
stop along the curve is the threshold decision, and it belongs to the business, not to this notebook.

✍️ **Deliverable 7 of 8 — The final test**

- **The final model**: its family and its hyperparameters: *...*
- **Its test score**, next to the cross-validated estimate and the fold spread: *...*
- **What the gap means**: *...*
- **Confirm the test set was used only here**, and that nothing was changed after seeing this score: *...*

💡 **Example** — Deliverable 7 for the churn data. Delete it before submitting.

- **Final model**: gradient boosting with `learning_rate` 0.037, 207 trees of depth 2, `subsample`
  0.91 and at least 24 rows per leaf, refitted on all 5,634 training rows with the Section 4
  preprocessing.
- **Test score**: 0.8455, against a cross-validated estimate of 0.8506 with a fold spread of 0.0135.
- **The gap**: 0.0051 lower on the test set, within one fold spread.
- **Test set use**: `X_test` appears only in the split and in this section. Nothing was changed after
  this score was computed.

## 8. Conclusions

🗑️ **Guide — Section 8.** Delete before submitting.

Write this section for a reader who reads **only this section**: a manager, or the team that would use
the model. Use plain words and the numbers you already have above. Do not introduce new results here.

🎯 **What we check:**
- A clear recommendation.
- Expected performance on new data, stated as an honest figure.
- At least two real limitations, one of them the untuned tournament.
- A concrete next step.

✍️ **Deliverable 8 of 8 — Conclusions**

- **Recommendation**: which model, for which decision: *...*
- **What to expect on new data**, in plain words: *...*
- **Limitations**, at least two: *...*
- **What you would do next**: *...*

💡 **Example** — Deliverable 8 for the churn data. Delete it before submitting.

- **Recommendation**: use the tuned gradient boosting model to rank customers by their risk of
  leaving, and give the retention team the list from the top down.
- **What to expect**: on customers the model has not seen, it ranks a customer who leaves above a
  customer who stays about 85% of the time (test AUC 0.8455).
- **Limitations**:
  1. The tournament compared models at starting settings, not tuned ones, and only gradient boosting
     was tuned. Logistic regression was 0.0016 behind it untuned and was never tuned.
  2. The data is one snapshot of customers. If prices or services change, the model must be checked
     again on new data.
  3. The model ranks customers but does not say how many to contact. That threshold needs the cost of
     an offer and the value of a customer kept.
- **Next step**: ask the business for those two costs and choose the threshold from them. Then check
  which columns drive the ranking before the retention team relies on it.

🗑️ **Before you submit** — delete this cell too.

1. Every ✍️ cell is answered about **your** data, and the bracketed header fields are filled in.
2. Every 💡 and 🗑️ cell is deleted, this one included.
3. Every markdown cell before and after a code cell describes **your** data and **your** numbers. No
   churn text remains.
4. Every number in your text appears in an output above it.
5. `X_test` appears only in Section 3 and Section 7.
6. *Restart Kernel and Run All Cells* finishes without an error, and you submit the file **with its
   outputs**.

**What comes next.** A model chosen this way is still a black box: we know how well it sorts, but not
why. The interpretability session starts from exactly such a model and asks which columns drive its
predictions.